# Chess · Deploy
Choose **Runtime → Run all** and allow Drive access. This notebook loads the saved champion, starts the API, opens a Cloudflare tunnel, tests it, and **finishes with your URL**. It never trains. If no champion exists yet, run **[train.ipynb](train.ipynb)** first.

A CPU runtime works for deployment; GPU is optional. Leave the Colab runtime connected while playing. This notebook does not prevent Colab idle/runtime limits. Code is collapsed but still executes normally.

Supports the original dense champion, evolved classical parameters and exported sparse neural/hybrid models. The saved bundle includes its engine code; after updating train.ipynb, run training/export again and restart this notebook to load that updated bundle. Existing published snapshots keep their original code.


In [ ]:
#@title 1 · Load the saved champion from Drive { display-mode: "form" }
# Standalone deployment: run this section after exporting, without rerunning training.
import sys, subprocess, importlib.util, json, time, threading
from pathlib import Path
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'fastapi', 'uvicorn', 'nest_asyncio', 'python-chess', 'numpy', 'requests'])
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
MODEL_ROOT = Path('/content/drive/MyDrive/cristopher_evo/models')
manifest_path = MODEL_ROOT / 'latest_champion.json'
if manifest_path.exists():
    import hashlib
    manifest = json.loads(manifest_path.read_text())
    API_MODELS = (MODEL_ROOT / manifest['folder']).resolve()
    if not API_MODELS.is_relative_to(MODEL_ROOT.resolve()):
        raise ValueError('Champion folder must be inside the models directory.')
    for name, checksum in manifest['sha256'].items():
        path = API_MODELS / name
        if path.parent != API_MODELS or not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest() != checksum:
            raise RuntimeError('Incomplete or damaged champion snapshot. Run train.ipynb to recover and publish again.')
else:
    API_MODELS = MODEL_ROOT  # Compatibility with exports made before the notebooks were split.
for name in ('cristopher_engine.py', 'cristopher_genome.json'):
    if not (API_MODELS / name).is_file():
        raise FileNotFoundError(f'Missing {name}. Run train.ipynb first to save a champion to Drive.')
api_metadata = json.loads((API_MODELS / 'cristopher_genome.json').read_text())
if api_metadata.get('weights'):
    if not (API_MODELS / api_metadata['weights']).is_file():
        raise FileNotFoundError('Missing exported neural weights. Run train.ipynb first to save the champion.')
    if Path(api_metadata['weights']).suffix == '.pt' and importlib.util.find_spec('torch') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])
# Re-running deployment first closes its previous server/tunnel, then loads the latest bundle.
if callable(globals().get('stop_tunnel')):
    stop_tunnel()
if callable(globals().get('stop_server')):
    stop_server()
spec = importlib.util.spec_from_file_location('cristopher_api_engine', API_MODELS / 'cristopher_engine.py')
API_MODULE = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = API_MODULE
spec.loader.exec_module(API_MODULE)
API_ENGINE = API_MODULE.CristopherEngine(API_MODELS / 'cristopher_genome.json')
API_VERSION = str(api_metadata.get('version', 'unknown'))
API_ENGINE_LOCK = threading.Lock()
if api_metadata.get('champion_generation') == -1:
    print('Notice: this is the initial saved baseline; training has not promoted a challenger yet.')
print(f'Loaded Chess ({API_VERSION}) from {API_MODELS}')
print('Search:', api_metadata.get('search_version', 'legacy export'))
print('Independent champion validation:', 'passed' if api_metadata.get('champion_verified') else 'not yet verified')
# No multipart dependency is needed: /move accepts JSON.
TUNNEL = 'cloudflare' #@param ['cloudflare', 'ngrok']
API_TIME_CAPS = {'easy': 3.0, 'medium': 10.0, 'hard': 25.0}


In [ ]:
#@title 2 · Prepare the API { display-mode: "form" }
import asyncio, math, sys
from collections import deque
from datetime import datetime, timezone
from typing import Literal
import chess
from fastapi import FastAPI, HTTPException, Request
from fastapi.exceptions import RequestValidationError
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field, field_validator
from starlette.concurrency import run_in_threadpool
from starlette.exceptions import HTTPException as StarletteHTTPException

# Stop the old app before replacing route globals on a rerun.
if callable(globals().get('stop_server')):
    stop_server()
API_STARTED = time.monotonic()
API_LOG = deque(maxlen=50)
API_COUNTERS = {'requests': 0, 'success': 0, 'errors': 0, 'fallbacks': 0}
API_STATS_LOCK = threading.Lock()

class MoveRequest(BaseModel):
    fen: str = Field(max_length=200)
    history: list[str] = Field(default_factory=list, max_length=2000)
    difficulty: Literal['easy', 'medium', 'hard'] = 'medium'

    @field_validator('fen')
    @classmethod
    def valid_fen(cls, value):
        try:
            board = chess.Board(value)
            if len(value.split()) != 6 or not board.is_valid():
                raise ValueError('invalid board')
        except ValueError:
            raise HTTPException(400, 'Invalid FEN: provide a valid six-field chess position.')
        return value

    @field_validator('history')
    @classmethod
    def valid_history(cls, value):
        for uci in value:
            try:
                move = chess.Move.from_uci(uci)
                if not move or move.drop:
                    raise ValueError('not a standard move')
            except ValueError:
                raise ValueError('History must contain standard UCI moves, such as e2e4.')
        return value

api = FastAPI(title='Cristopher API', version=API_VERSION)

@api.exception_handler(StarletteHTTPException)
async def http_error(_request, exc):
    return JSONResponse(status_code=exc.status_code, content={'error': str(exc.detail)})

@api.exception_handler(RequestValidationError)
async def validation_error(_request, exc):
    message = '; '.join(f"{'.'.join(map(str, e['loc']))}: {e['msg']}" for e in exc.errors())
    return JSONResponse(status_code=422, content={'error': message})

@api.middleware('http')
async def record_request(request: Request, call_next):
    if request.method != 'POST' or request.url.path != '/move':
        return await call_next(request)
    started = time.monotonic()
    entry = {'fen': None, 'difficulty': None, 'move': None}
    try:
        body = await request.json()
        if isinstance(body, dict):
            entry.update(fen=body.get('fen'), difficulty=body.get('difficulty', 'medium'))
    except (ValueError, UnicodeDecodeError):
        pass
    try:
        response = await call_next(request)
    except Exception:
        response = JSONResponse(status_code=500, content={'error': 'Engine failed. Retry or check the notebook.'})
    entry.update(getattr(request.state, 'move_log', {}))
    entry.update(status=response.status_code, time_ms=int((time.monotonic() - started) * 1000),
                 at=datetime.now(timezone.utc).isoformat())
    with API_STATS_LOCK:
        API_LOG.append(entry)
        API_COUNTERS['requests'] += 1
        API_COUNTERS['success' if response.status_code < 400 else 'errors'] += 1
        API_COUNTERS['fallbacks'] += int(entry.get('fallback', False))
    return response

# Outermost middleware also adds CORS headers to errors, including custom headers.
api.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@api.get('/health')
async def health():
    return {'status': 'ok', 'engine': 'cristopher', 'version': API_VERSION,
            'uptime_s': int(time.monotonic() - API_STARTED)}

@api.get('/stats')
async def stats():
    with API_STATS_LOCK:
        return {'counters': dict(API_COUNTERS), 'last_requests': list(API_LOG)}

class ApiSearchDeadline(Exception):
    pass

def depth_one_fallback(board):
    """Score every legal successor with the exported evaluation; no random fallback."""
    moves = list(board.legal_moves)
    children = []
    for move in moves:
        child = board.copy(stack=True)
        child.push(move)
        children.append(child)
    if hasattr(API_ENGINE, 'values'):  # GPU export: batched neural evaluation.
        scores = [-float(value) for value in API_ENGINE.values(children)]
    else:
        genome = API_ENGINE.nn if API_ENGINE.nn is not None else API_ENGINE.genome
        scores = [-float(API_MODULE.evaluate(child, genome)) for child in children]
    for i, child in enumerate(children):
        if child.is_checkmate():
            scores[i] = 30000.0
        elif child.is_game_over():
            scores[i] = 0.0
    best = max(range(len(moves)), key=lambda i: scores[i])
    return {'move': moves[best].uci(), 'info': {'eval': scores[best], 'depth': 1,
            'nodes': len(moves), 'fallback': True}}

def search_locked(payload, board, deadline):
    # Waiting for the lock counts against the request budget too.
    remaining = deadline - time.monotonic()
    if remaining <= 0 or not API_ENGINE_LOCK.acquire(timeout=max(0, remaining)):
        raise ApiSearchDeadline()
    old_trace = sys.gettrace()
    limits = API_ENGINE.settings[payload.difficulty]
    old_seconds = limits['seconds']
    def check_deadline(frame, event, arg):
        if frame.f_code is not search_locked.__code__ and time.monotonic() >= deadline:
            raise ApiSearchDeadline()
        return check_deadline
    try:
        limits['seconds'] = min(float(old_seconds), max(.001, deadline - time.monotonic()))
        # Existing search deadlines plus a Python deadline cover preprocessing/fallback.
        # Native NumPy/Torch calls finish before Python can observe this deadline.
        sys.settrace(check_deadline)
        try:
            result = API_ENGINE.get_move(payload.fen, payload.history, payload.difficulty)
        except AssertionError:
            result = {}  # Export assertions may catch an illegal move before returning it.
        try:
            move = chess.Move.from_uci(result.get('move', ''))
        except (ValueError, TypeError, AttributeError):
            move = None
        if move not in board.legal_moves:
            result = depth_one_fallback(board)
        info = result.get('info', {})
        score = float(info.get('eval', 0))
        if not math.isfinite(score):
            raise ValueError('Engine returned a non-finite score')
        # Exports score in side-to-move centipawns. HTTP scores are White pawns.
        info = {**info, 'eval': score * (1 if board.turn == chess.WHITE else -1) / 100,
                'depth': int(info.get('depth', 0)), 'nodes': int(info.get('nodes', 0))}
        return {'move': result['move'], 'info': info}
    finally:
        sys.settrace(old_trace)
        limits['seconds'] = old_seconds
        API_ENGINE_LOCK.release()

@api.post('/move')
async def move(payload: MoveRequest, request: Request):
    board = chess.Board(payload.fen)
    if payload.history:
        replay = chess.Board()
        try:
            for uci in payload.history:
                replay.push_uci(uci)
        except ValueError:
            raise HTTPException(400, 'Invalid history: moves must be legal from the starting position.')
        if replay.fen() != board.fen():
            raise HTTPException(400, 'History does not match the supplied FEN.')
        board = replay
    if board.is_game_over() or board.is_repetition(3) or board.is_fifty_moves():
        raise HTTPException(400, 'Game over: this position has no move to play.')
    started = time.monotonic()
    cap = API_TIME_CAPS[payload.difficulty]
    task = asyncio.create_task(run_in_threadpool(search_locked, payload, board, started + cap))
    # Keep the lock in the worker even if HTTP times out: never overlap engine calls.
    done, _ = await asyncio.wait({task}, timeout=cap)
    if not done:
        # Consume a late worker exception. The trace stops Python work on its next line.
        task.add_done_callback(lambda t: t.exception() if not t.cancelled() else None)
        raise HTTPException(504, f'Cristopher exceeded the {cap:g}s {payload.difficulty} time cap. Retry.')
    try:
        result = task.result()
    except ApiSearchDeadline:
        raise HTTPException(504, f'Cristopher exceeded the {cap:g}s {payload.difficulty} time cap. Retry.')
    except Exception:
        raise HTTPException(500, 'Engine failed. Retry or check the exported model files.')
    # Verify again at the API boundary before returning anything to the browser.
    if chess.Move.from_uci(result['move']) not in board.legal_moves:
        raise HTTPException(500, 'Engine could not produce a legal move.')
    result['info']['time_ms'] = int((time.monotonic() - started) * 1000)
    request.state.move_log = {'move': result['move'], 'fallback': result['info'].get('fallback', False)}
    return result

print('API ready: /health, /move, /stats')


In [ ]:
#@title 3 · Start the API and run its self-tests { display-mode: "form" }
import requests, uvicorn, nest_asyncio
nest_asyncio.apply()

def stop_server():
    server = globals().get('API_SERVER')
    thread = globals().get('API_SERVER_THREAD')
    if server is not None:
        server.should_exit = True
    if thread is not None and thread.is_alive():
        thread.join(timeout=5)
        if thread.is_alive():
            server.force_exit = True
            thread.join(timeout=2)
        if thread.is_alive():
            raise RuntimeError('Previous server is still stopping. Wait and rerun this cell.')
    # A timed-out native call can still own the engine; do not replace its lock.
    if not API_ENGINE_LOCK.acquire(timeout=2):
        raise RuntimeError('Engine is still finishing a request. Wait and rerun this cell.')
    API_ENGINE_LOCK.release()

stop_server()
API_SERVER = uvicorn.Server(uvicorn.Config(api, host='0.0.0.0', port=8000,
                           log_level='warning', loop='asyncio', timeout_graceful_shutdown=3))
API_SERVER_THREAD = threading.Thread(target=API_SERVER.run, daemon=True, name='cristopher-api')
API_SERVER_THREAD.start()
for _ in range(100):
    if API_SERVER.started:
        break
    if not API_SERVER_THREAD.is_alive():
        raise RuntimeError('Server failed to start. Port 8000 may already be in use.')
    time.sleep(.1)
else:
    stop_server()
    raise RuntimeError('Server did not become ready on port 8000.')

def api_self_test(base_url, public=False):
    headers = {'ngrok-skip-browser-warning': 'true'}
    health_response = requests.get(base_url + '/health', headers=headers, timeout=4)
    health_response.raise_for_status()
    assert health_response.json()['status'] == 'ok'
    response = requests.post(base_url + '/move', headers=headers, timeout=15,
        json={'fen': chess.STARTING_FEN, 'history': [], 'difficulty': 'easy'})
    response.raise_for_status()
    assert chess.Move.from_uci(response.json()['move']) in chess.Board().legal_moves
    print('OK: health + legal starting-position move' + (' through PUBLIC URL' if public else ' locally'))
    if not public:
        for fen, label in [('not a FEN', 'invalid FEN'), ('7k/5Q2/6K1/8/8/8/8/8 b - - 0 1', 'finished game')]:
            response = requests.post(base_url + '/move', headers=headers, timeout=4,
                                     json={'fen': fen, 'difficulty': 'medium'})
            assert response.status_code == 400, response.text
            assert response.json().get('error'), response.text
            print(f'OK: {label} -> 400: {response.json()["error"]}')

api_self_test('http://127.0.0.1:8000')
print('Server running in background on port 8000. Use stop_server() to stop it.')


def shutdown_api():
    try:
        if callable(globals().get('stop_tunnel')):
            stop_tunnel()
    finally:
        stop_server()
    print('API and tunnel stopped.')


## Use the URL
The last cell prints the verified HTTPS URL. In the game, open **Settings → Server URL**, paste it, enable **Use AI server**, and test the connection. Select AI and press Play. Restarting deployment can change the URL.

To stop: disconnect this runtime, or add a cell and call `shutdown_api()`. **Run all leaves the server running**; there is no blocking heartbeat or automatic shutdown cell. To inspect request counters, open `<URL>/stats`.

Missing files: run the training notebook. Tunnel failure: Run all again after checking the runtime network. CORS/mixed content: use the printed HTTPS base URL, without `/move`. The API retains legal-move validation, request time caps, locking, and its 50-request log.


In [ ]:
#@title 4 · Open the tunnel and print the verified URL { display-mode: "form" }
import re, queue, urllib.request, html
from IPython.display import display, HTML

def stop_tunnel():
    global PUBLIC_URL
    process = globals().get('CLOUDFLARED_PROCESS')
    if process is not None:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=3)
        reader = globals().get('CLOUDFLARED_READER')
        if reader is not None:
            reader.join(timeout=2)
        if process.stdout:
            process.stdout.close()
    tunnel = globals().get('NGROK_TUNNEL')
    if tunnel is not None:
        from pyngrok import ngrok
        ngrok.disconnect(tunnel.public_url)
        ngrok.kill()
        globals()['NGROK_TUNNEL'] = None
    PUBLIC_URL = None

stop_tunnel()
try:
    if TUNNEL == 'cloudflare':
        binary = Path('/content/cloudflared')
        if not binary.exists():
            # Official Linux amd64 release for standard Colab runtimes.
            download = binary.with_suffix('.download')
            urllib.request.urlretrieve(
                'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', download)
            download.chmod(0o755)
            download.replace(binary)
        tunnel_lines = queue.Queue()
        tunnel_tail = deque(maxlen=30)
        CLOUDFLARED_PROCESS = subprocess.Popen(
            [str(binary), 'tunnel', '--url', 'http://localhost:8000'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        def read_tunnel_output(process, output, tail):
            # Drain continuously so cloudflared cannot block on a full output pipe.
            for line in process.stdout:
                tail.append(line.rstrip())
                match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com\b', line)
                if match:
                    output.put(match.group(0))
        CLOUDFLARED_READER = threading.Thread(target=read_tunnel_output,
            args=(CLOUDFLARED_PROCESS, tunnel_lines, tunnel_tail), daemon=True)
        CLOUDFLARED_READER.start()
        try:
            PUBLIC_URL = tunnel_lines.get(timeout=45)
        except queue.Empty:
            raise RuntimeError('Cloudflare did not print a URL. Check network/config:\n' + '\n'.join(tunnel_tail))
    elif TUNNEL == 'ngrok':
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'])
        from pyngrok import ngrok, conf
        from getpass import getpass
        try:
            from google.colab import userdata
            token = userdata.get('NGROK_TOKEN')
        except Exception:
            token = None
        if not token:
            token = getpass('Ngrok auth token (hidden): ')
        # Pass in-memory configuration; never print or hardcode the secret.
        ngrok_config = conf.PyngrokConfig(auth_token=token)
        try:
            NGROK_TUNNEL = ngrok.connect(8000, 'http', bind_tls=True, pyngrok_config=ngrok_config)
        except Exception:
            # Do not echo provider exceptions that could contain credentials.
            raise RuntimeError('Ngrok failed. Check NGROK_TOKEN and your ngrok account.') from None
        finally:
            token = None
            ngrok_config.auth_token = None
        PUBLIC_URL = NGROK_TUNNEL.public_url
    else:
        raise ValueError('TUNNEL must be cloudflare or ngrok')
    public_ok = False
    for attempt in range(6):
        try:
            api_self_test(PUBLIC_URL, public=True)
            public_ok = True
            break
        except (requests.RequestException, ValueError, AssertionError) as error:
            print(f'Public test {attempt + 1}/6: FAIL ({type(error).__name__}); retrying…')
            time.sleep(3)
    if not public_ok:
        raise RuntimeError('Public endpoint tests failed. Run all again to retry deployment.')
    display(HTML(f'<h2>API ready</h2><p><strong>{html.escape(PUBLIC_URL)}</strong></p>'))
    print(f'Paste this URL in the web page -> Connect to Cristopher: {PUBLIC_URL}')
except BaseException:
    stop_tunnel()
    raise
